# Анализ завершённых экспериментов

Ноутбук строит графики только из файлов, сохранённых в `runs/`. Для сравнения моделей добавь в `RUN_DIRS` пути к папкам U-Net, FPN и DeepLabV3+ после их test-оценки.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

# В Kaggle подключи Output завершённых запусков как Dataset Input.
RUNS_ROOT = Path('/kaggle/input/landcover-experiment-runs/runs')
available_runs = sorted(path for path in RUNS_ROOT.iterdir() if path.is_dir())
for path in available_runs:
    print(path.name)

In [ ]:
# Для первой визуализации выбери папку U-Net.
RUN_DIR = available_runs[0]  # замени на явный путь после появления нескольких запусков
history = pd.read_csv(RUN_DIR / 'history.csv')
best_row = history.loc[history['val_foreground_miou'].idxmax()]
print('Run:', RUN_DIR.name)
print('Best epoch:', int(best_row['epoch']))
print('Best validation foreground mIoU:', f"{best_row['val_foreground_miou']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['epoch'], history['train_loss'], label='train loss')
axes[0].plot(history['epoch'], history['val_loss'], label='validation loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Динамика loss')
axes[0].legend()

axes[1].plot(history['epoch'], history['val_foreground_miou'], marker='o', label='validation foreground mIoU')
axes[1].axvline(best_row['epoch'], color='tab:red', linestyle='--', label='best epoch')
axes[1].set(xlabel='Epoch', ylabel='foreground mIoU', title='Критерий выбора checkpoint')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Выполняется после test evaluation модели.
test_metrics = pd.read_csv(RUN_DIR / 'test_metrics.csv').iloc[0]
per_scene = pd.read_csv(RUN_DIR / 'test_per_scene_metrics.csv').sort_values('foreground_miou')

print('Test foreground mIoU:', f"{test_metrics['foreground_miou']:.4f}")
display(per_scene[['scene', 'foreground_miou', 'iou_buildings', 'iou_woodland', 'iou_water', 'iou_roads']])

ax = per_scene.plot.barh(x='scene', y='foreground_miou', legend=False, figsize=(8, 4), color='tab:blue')
ax.set(xlabel='foreground mIoU', ylabel='Test scene', title='Качество U-Net по независимым территориям')
plt.tight_layout()
plt.show()

## Сравнение архитектур

Раскомментируй и заполни `RUN_DIRS` после завершения трёх честных test-оценок.

In [ ]:
# RUN_DIRS = [
#     RUNS_ROOT / '..._unet_resnet18_scratch',
#     RUNS_ROOT / '..._fpn_resnet18_scratch',
#     RUNS_ROOT / '..._deeplabv3plus_resnet18_scratch',
# ]
# 
# rows = []
# for run_dir in RUN_DIRS:
#     config = json.loads((run_dir / 'summary.json').read_text())
#     metrics = pd.read_csv(run_dir / 'test_metrics.csv').iloc[0].to_dict()
#     rows.append({'run': run_dir.name, **config, **metrics})
# comparison = pd.DataFrame(rows).sort_values('foreground_miou', ascending=False)
# display(comparison)
# comparison.to_csv('/kaggle/working/architecture_comparison.csv', index=False)